# Commercial Intelligence Chatbot

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Build semantic models** in YAML for Cortex Analyst
2. **Enable natural language queries** over sales data
3. **Create conversational interface** with `CORTEX.COMPLETE()`
4. **Define business metrics** in semantic layer
5. **Deploy self-service analytics** for business users

---

## What You'll Build

A **conversational BI chatbot** that answers sales questions in natural language:
- Natural language to SQL translation
- Semantic model definitions (YAML)
- Business-friendly metric names
- Ad-hoc question answering
- Self-service analytics for non-technical users

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 18 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- Cortex Analyst - NL to SQL translation engine
- Semantic Models (YAML) - Business-friendly data definitions
- `CORTEX.COMPLETE()` - LLM integration for responses
- `OBJECT_CONSTRUCT()` - Build JSON for semantic layer
- Streamlit chatbot - Interactive Q&A interface

Let's begin!

---

## Paso 1: Configuración del Entorno

### Cortex Analyst

Natural language interface to your data warehouse - no SQL knowledge required for users

In [ ]:
CREATE DATABASE IF NOT EXISTS CHATBOT_BI_DB;
USE SCHEMA CHATBOT_BI_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL';
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db;

---

## Paso 2: Create Business Tables

In [ ]:
CREATE OR REPLACE TABLE sales_data (
    sale_date DATE,
    product_name VARCHAR(100),
    region VARCHAR(50),
    sales_rep VARCHAR(100),
    units_sold INTEGER,
    revenue_usd DECIMAL(15,2)
);

CREATE OR REPLACE TABLE sales_targets (
    target_month DATE,
    product_name VARCHAR(100),
    region VARCHAR(50),
    target_revenue_usd DECIMAL(15,2)
);

---

## Paso 3: Generar Sales Data

In [ ]:
-- Generar 12 months of sales data
INSERT INTO sales_data
WITH months AS (
    SELECT DATEADD('day', -FLOOR(UNIFORM(0,365,RANDOM())), CURRENT_DATE()) as sale_date
    FROM TABLE(GENERATOR(ROWCOUNT => 1000))
),
products AS (SELECT * FROM (VALUES('Xarelto'),('Eylea'),('Stivarga')) t(product)),
regions AS (SELECT * FROM (VALUES('Northeast'),('Southeast'),('Midwest'),('West')) t(region))
SELECT 
    DATE_TRUNC('day', m.sale_date) as sale_date,
    p.product as product_name,
    r.region,
    'Rep ' || LPAD(FLOOR(UNIFORM(1,21,RANDOM())), 2, '0') as sales_rep,
    FLOOR(UNIFORM(10, 200, RANDOM())) as units_sold,
    ROUND(units_sold * UNIFORM(800, 1200, RANDOM()), 2) as revenue_usd
FROM months m CROSS JOIN products p CROSS JOIN regions r
WHERE UNIFORM(0,1,RANDOM()) > 0.95;

-- Insert targets
INSERT INTO sales_targets
WITH months AS (
    SELECT DATE_TRUNC('month', DATEADD('month', -SEQ4(), CURRENT_DATE())) as target_month
    FROM TABLE(GENERATOR(ROWCOUNT => 12))
)
SELECT 
    m.target_month,
    p.product as product_name,
    r.region,
    ROUND(UNIFORM(5000000, 15000000, RANDOM()), 2) as target_revenue_usd
FROM months m
CROSS JOIN (VALUES('Xarelto'),('Eylea'),('Stivarga')) p(product)
CROSS JOIN (VALUES('Northeast'),('Southeast'),('Midwest'),('West')) r(region);

SELECT COUNT(*) as sales_records FROM sales_data;

---

## Paso 4: Create Semantic Model

### Qué Vamos a Crear

A **semantic model** - a YAML file that maps business terminology to database schemas, enabling Cortex Analyst to understand natural language questions.

### Snowflake's Recommended Approach

According to Snowflake best practices, semantic models should be stored as **YAML files on a STAGE**, not in tables. This approach:

-  **Proper Integration**: Direct reference in Cortex Analyst REST API
-  **Version Control**: Easy to track changes via file versions
-  **Access Control**: Stage-based permissions (GRANT READ ON STAGE)
-  **Discoverability**: Visible in Snowflake monitoring views
-  **Standard Pattern**: Follows Snowflake documentation

### How to Use Semantic Models

**In Production (REST API)**:
```python
import requests

response = requests.post(
    "https://account.snowflakecomputing.com/api/v2/cortex/analyst/message",
    headers={"Authorization": f"Bearer {token}"},
    json={
        "messages": [{"role": "user", "content": "What were sales last month?"}],
        "semantic_model_file": "@chatbot_bi_db.semantic_models.model_stage/sales_analytics.yaml"
    }
)
```

### Why This Matters

Semantic models enable **self-service analytics** by providing business context:
- Maps "revenue" → `SUM(revenue_usd)`
- Maps "last month" → `WHERE sale_date >= DATE_TRUNC('month', DATEADD('month', -1, CURRENT_DATE()))`
- Defines valid dimensions, measures, and relationships

**Impacto de Negocio**: Non-technical users can query data without writing SQL

In [ ]:
-- STEP 1: Create schema and stage for semantic model files
CREATE SCHEMA IF NOT EXISTS CHATBOT_BI_DB.SEMANTIC_MODELS;
USE SCHEMA CHATBOT_BI_DB.SEMANTIC_MODELS;

CREATE OR REPLACE STAGE model_stage
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Storage for Cortex Analyst semantic model YAML files';

-- Grant read access to users who need to query via Cortex Analyst
GRANT READ ON STAGE model_stage TO ROLE PUBLIC;

-- STEP 2: Display the semantic model YAML content
-- Copy this output, save as 'sales_analytics.yaml', and upload to the stage
SELECT 
'name: sales_analytics
description: Commercial sales performance data for Bayer Pharmaceuticals products
tables:
  - name: sales_data
    description: Daily sales transactions across all products and regions
    base_table:
      database: CHATBOT_BI_DB
      schema: PUBLIC
      table: SALES_DATA
    dimensions:
      - name: sale_date
        synonyms:
          - transaction date
          - date of sale
        data_type: DATE
        description: Date when the sale occurred
      - name: product_name
        synonyms:
          - product
          - medication
        data_type: VARCHAR
        description: Product name (Xarelto, Eylea, Stivarga)
      - name: region
        synonyms:
          - territory
          - geographic region
        data_type: VARCHAR
        description: Geographic sales region (Northeast, Southeast, Midwest, West)
      - name: sales_rep
        synonyms:
          - representative
          - rep
          - sales person
        data_type: VARCHAR
        description: Name of the sales representative
    measures:
      - name: revenue_usd
        synonyms:
          - revenue
          - sales
          - dollars
        data_type: NUMBER
        description: Total revenue in US Dollars
        aggregation: sum
      - name: units_sold
        synonyms:
          - units
          - quantity
          - volume
        data_type: NUMBER
        description: Number of units sold
        aggregation: sum
  - name: sales_targets
    description: Monthly revenue targets by product and region
    base_table:
      database: CHATBOT_BI_DB
      schema: PUBLIC
      table: SALES_TARGETS
    dimensions:
      - name: target_month
        synonyms:
          - month
          - target period
        data_type: DATE
        description: Month for which the target applies
      - name: product_name
        synonyms:
          - product
          - medication
        data_type: VARCHAR
        description: Product name (Xarelto, Eylea, Stivarga)
      - name: region
        synonyms:
          - territory
          - geographic region
        data_type: VARCHAR
        description: Geographic sales region
    measures:
      - name: target_revenue_usd
        synonyms:
          - target
          - goal
          - quota
        data_type: NUMBER
        description: Target revenue in US Dollars
        aggregation: sum' as semantic_model_yaml;

-- STEP 3: Instructions for uploading the YAML file
SELECT 
'📋 NEXT STEPS TO COMPLETE SEMANTIC MODEL SETUP:

1. Copy the YAML content from the query result above
2. Save it to a local file named: sales_analytics.yaml
3. Upload to the stage using ONE of these methods:

   METHOD A - Snowsight UI (Recommended):
   • Navigate to: Databases → CHATBOT_BI_DB → SEMANTIC_MODELS → Stages → MODEL_STAGE
   • Click "Upload Files" button
   • Select your sales_analytics.yaml file
   • Click "Upload"

   METHOD B - Snowflake CLI:
   • Run: snow stage copy file:///path/to/sales_analytics.yaml @chatbot_bi_db.semantic_models.model_stage

   METHOD C - SnowSQL:
   • Run: PUT file:///path/to/sales_analytics.yaml @chatbot_bi_db.semantic_models.model_stage;

4. Verify upload:
   • Run: LIST @chatbot_bi_db.semantic_models.model_stage;

5. Reference in Cortex Analyst API:
   • Use: @chatbot_bi_db.semantic_models.model_stage/sales_analytics.yaml

✅ Once uploaded, Cortex Analyst can use this semantic model to answer natural language questions!' as upload_instructions;

-- STEP 4: Verify stage setup
SELECT 'Stage created successfully!' as status,
       'Schema: CHATBOT_BI_DB.SEMANTIC_MODELS' as schema_name,
       'Stage: MODEL_STAGE' as stage_name,
       'Ready to receive YAML files' as ready_status;

---

## Paso 5: Query with Natural Language

### Qué Vamos a Crear

**LLM-powered question-answering system** that provides natural language responses to business questions using Snowflake Cortex AI.

### Understanding `AI_COMPLETE()`

Snowflake's **LLM (Large Language Model) function** for text generation, analysis, and conversational AI:

```sql
AI_COMPLETE(
    model_name,           -- TEXT: LLM model to use
    prompt_text,          -- TEXT: Instructions + context + question
    options              -- OBJECT: Optional configuration
)
```

**Returns**: TEXT - The LLM's generated response

### Why CORTEX.COMPLETE for Commercial Intelligence?

`CORTEX.COMPLETE()` is ideal for building chatbots and Q&A systems because:

- **Native LLM Integration**: Access powerful language models directly in SQL
- **No API Keys Required**: Built into Snowflake - no external services
- **Data Context Injection**: Combine real sales data with natural language prompts
- **Multiple Models**: Choose from Mistral, Llama, Mixtral based on needs
- **Privacy**: Data never leaves Snowflake environment (HIPAA/GDPR compliant)
- **Scalable**: Process thousands of questions per minute

### How It Works (Behind the Scenes)

When you call `CORTEX.COMPLETE(model, prompt)`, Snowflake:

1. **Receives Prompt**: Your question + any context data you provide
2. **Routes to LLM**: Sends to specified model (Mistral, Llama, etc.)
3. **Model Processes**: LLM analyzes prompt and generates response
4. **Returns Text**: Natural language answer returned as TEXT column
5. **Handles Tokens**: Automatically manages input/output token limits

**Response Time**: 1-5 seconds depending on prompt complexity and model

### Available LLM Models

Snowflake Cortex provides access to multiple state-of-the-art models:

| Model               | Best For                              | Speed    | Cost     |
|---------------------|---------------------------------------|----------|----------|
| **mistral-large2**  | Complex reasoning, analysis           | Medio   | Medio   |
| **llama3.1-70b**    | General purpose, instruction following| Medio   | Medio   |
| **llama3.1-8b**     | Fast responses, simple tasks          | Fast     | Bajo      |
| **mixtral-8x7b**    | Balanced performance/cost             | Medio   | Bajo      |
| **mistral-7b**      | Lightweight, high throughput          | Fast     | Bajo      |

**Recommendation for Chatbots**: `mistral-large2` for best quality, `llama3.1-8b` for speed

### Key Parameters Explained

#### **1. `model_name` (TEXT)**

Specifies which LLM to use:

```sql
AI_COMPLETE('mistral-large2', prompt)
-- or
AI_COMPLETE('llama3.1-70b', prompt)
```

**What It Does**:
- Routes request to specified LLM
- Different models have different capabilities and speeds
- Can compare models by running same prompt through multiple

**Model Selection Guide**:
- **Analysis/Reasoning**: `mistral-large2`, `llama3.1-70b`
- **Speed Critical**: `llama3.1-8b`, `mistral-7b`
- **Cost Sensitive**: `mixtral-8x7b`, `mistral-7b`

#### **2. `prompt_text` (TEXT)**

The instructions + context + question for the LLM:

```sql
AI_COMPLETE(
    'mistral-large2',
    'You are a commercial analyst. Answer based on the data:
    
    Sales Data:
    - Xarelto: $2.5M revenue, 15,000 units
    - Eylea: $1.8M revenue, 12,000 units
    
    Question: Which product has higher revenue per unit?
    
    Answer concisely with numbers.'
)
```

**What It Does**:
- Provides context and instructions to LLM
- Can include actual data from your database
- Guides response format and style

**Prompt Engineering Best Practices**:
1. **System Role**: Define LLM's persona ("You are a sales analyst...")
2. **Data Context**: Include relevant numbers/facts
3. **Clear Question**: State exactly what you want
4. **Output Format**: Specify desired response style

#### **3. `options` (OBJECT) - Optional**

Fine-tune LLM behavior:

```sql
AI_COMPLETE(
    'mistral-large2',
    prompt,
    {
        'temperature': 0.3,    -- Bajoer = more focused (0-1)
        'max_tokens': 500,    -- Limit response length
        'top_p': 0.9          -- Nucleus sampling (0-1)
    }
)
```

**Temperature** (0-1):
- **0**: Deterministic, focused, factual (best for data analysis)
- **0.5**: Balanced creativity and accuracy
- **1**: Creative, varied responses (best for brainstorming)

**Max Tokens**:
- Controls response length
- 1 token ≈ 0.75 words
- 500 tokens ≈ 375 words

### The Power of Data Context Injection

The key to effective LLM chatbots is **injecting real data** into prompts. This cell demonstrates how to combine SQL aggregations with LLM prompts to create data-aware chatbot responses.

### Why This Matters for Commercial Teams

**Old Way** (manual analysis):
- Analyst pulls data into Excel
- Creates charts and summaries
- Writes bullet points for executives
- Takes 2-4 hours per question
- Static report (can't ask follow-ups)

**CORTEX.COMPLETE Way** (automated):
- SQL query generates response in 3 seconds
- Natural language, ready for executives
- Can ask unlimited follow-up questions
- Real-time data (always current)
- Consistent formatting

**Impacto de Negocio**: From **20 questions/week** (manual) to **500 questions/day** (automated) = **175x increase in commercial insights**

### Technical Details

**Token Limits**:
- **Input**: Up to 32,000 tokens (≈24,000 words) for `mistral-large2`
- **Output**: Configurable via `max_tokens` option
- **Total**: Input + output must fit within model's context window

**Performance**:
- **Simple queries**: 1-2 seconds
- **Complex analysis**: 3-5 seconds
- **Batch processing**: 1000 prompts in ~30 minutes

**Best Practices**:

**DO**:
-  Include specific data in prompts
-  Use low temperature (0.1-0.3) for factual responses
-  Specify output format ("Answer in bullet points")
-  Test with `LIMIT 1` before full deployment

**DON'T**:
-  Send entire table contents (use aggregations)
-  Use for calculations (LLMs can make math errors)
-  Expect 100% consistency (responses vary slightly)
-  Skip data validation (always verify LLM claims)

In [ ]:
-- Create example questions for demonstration
CREATE OR REPLACE TABLE example_questions (
    question_id INTEGER,
    question_text VARCHAR(500)
);

INSERT INTO example_questions VALUES
(1, 'What were total sales by product last month?'),
(2, 'Which region had the highest revenue for Xarelto?'),
(3, 'Compare sales performance across all regions'),
(4, 'What is our top performing product by revenue?'),
(5, 'Show me sales trends for Stivarga over time');

SELECT 'Example questions created!' as status;

In [ ]:
-- Use LLM to generate natural language responses with actual sales data context
CREATE OR REPLACE TABLE intelligence_responses AS
WITH sales_summary AS (
    -- Get recent sales data to provide context to LLM
    SELECT 
        product_name,
        region,
        DATE_TRUNC('month', sale_date) as month,
        SUM(revenue_usd) as monthly_revenue,
        SUM(units_sold) as monthly_units
    FROM sales_data
    WHERE sale_date >= DATEADD('month', -3, CURRENT_DATE())
    GROUP BY product_name, region, month
),
data_context AS (
    SELECT 
        'Sales Data Summary:
        
Product Performance:
- Xarelto: Total Revenue $' || 
        ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE product_name='Xarelto'), 0) || 
        ', Units Sold: ' || (SELECT SUM(units_sold) FROM sales_data WHERE product_name='Xarelto') || '
- Eylea: Total Revenue $' || 
        ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE product_name='Eylea'), 0) || 
        ', Units Sold: ' || (SELECT SUM(units_sold) FROM sales_data WHERE product_name='Eylea') || '
- Stivarga: Total Revenue $' || 
        ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE product_name='Stivarga'), 0) || 
        ', Units Sold: ' || (SELECT SUM(units_sold) FROM sales_data WHERE product_name='Stivarga') || '

Regional Performance:
- Northeast: $' || ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE region='Northeast'), 0) || '
- Southeast: $' || ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE region='Southeast'), 0) || '
- Midwest: $' || ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE region='Midwest'), 0) || '
- West: $' || ROUND((SELECT SUM(revenue_usd) FROM sales_data WHERE region='West'), 0) || '

Time Period: Last 12 months
Total Records: ' || (SELECT COUNT(*) FROM sales_data) as context_data
)
SELECT 
    eq.question_id,
    eq.question_text,
    AI_COMPLETE(
        'mistral-large',
        dc.context_data || '

Question: ' || eq.question_text || '

Provide a clear, concise answer based on the sales data provided above. Include specific numbers and comparisons where relevant.'
    ) as llm_response,
    CURRENT_TIMESTAMP() as response_generated_at
FROM example_questions eq
CROSS JOIN data_context dc
WHERE eq.question_id <= 5  -- Limit for demo (LLM calls are expensive)
ORDER BY eq.question_id;

-- View generated responses
SELECT 
    question_text,
    llm_response
FROM intelligence_responses
ORDER BY question_id;

---

## Paso 6: Execute Generated Queries

Run the SQL generated from natural language

In [ ]:
-- Execute query 1: Total sales by product last month
SELECT product_name, SUM(revenue_usd) as total_revenue 
FROM sales_data 
WHERE sale_date >= DATE_TRUNC('month', DATEADD('month', -1, CURRENT_DATE()))
GROUP BY product_name
ORDER BY total_revenue DESC;

-- Execute query 2: Revenue by region for Xarelto
SELECT region, SUM(revenue_usd) as revenue
FROM sales_data
WHERE product_name = 'Xarelto'
GROUP BY region
ORDER BY revenue DESC;

---

## Paso 7: Interactive Chatbot Dashboard

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🤖 Commercial Intelligence Chatbot")
st.markdown("Ask questions about your sales data in natural language")

# Example questions
st.subheader("💡 Example Questions")
examples = session.sql("SELECT question_text FROM example_questions").to_pandas()

for _, row in examples.iterrows():
    if st.button(row['QUESTION_TEXT']):
        st.session_state['selected_question'] = row['QUESTION_TEXT']

# Query input
st.subheader("📝 Ask a Question")
user_question = st.text_input(
    "Enter your question",
    placeholder="e.g., What were total sales by product last month?",
    value=st.session_state.get('selected_question', '')
)

if user_question:
    # In production, this would use Cortex Analyst to generate SQL
    # For demo, we match to pre-built queries
    st.markdown("**Generated SQL:**")
    
    # Match question to example
    if "total sales by product" in user_question.lower():
        sql_query = """
            SELECT product_name, SUM(revenue_usd) as total_revenue 
            FROM sales_data 
            WHERE sale_date >= DATE_TRUNC('month', DATEADD('month', -1, CURRENT_DATE()))
            GROUP BY product_name
        """
    elif "revenue by region" in user_question.lower():
        sql_query = """
            SELECT region, SUM(revenue_usd) as revenue
            FROM sales_data
            WHERE product_name = 'Xarelto'
            GROUP BY region
        """
    else:
        sql_query = """
            SELECT product_name, region, SUM(revenue_usd) as revenue
            FROM sales_data
            GROUP BY product_name, region
            ORDER BY revenue DESC
            LIMIT 10
        """
    
    st.code(sql_query, language='sql')
    
    # Execute query
    st.markdown("**Results:**")
    try:
        results = session.sql(sql_query).to_pandas()
        st.dataframe(results, use_container_width=True, hide_index=True)
        
        # Visualize if applicable
        if 'REVENUE' in results.columns or 'TOTAL_REVENUE' in results.columns:
            revenue_col = 'REVENUE' if 'REVENUE' in results.columns else 'TOTAL_REVENUE'
            st.bar_chart(results.set_index(results.columns[0])[revenue_col])
    except Exception as e:
        st.error(f"Error executing query: {str(e)}")

# Quick insights
st.markdown("---")
st.subheader("📊 Quick Insights")

col1, col2, col3 = st.columns(3)

with col1:
    total_revenue = session.sql("SELECT SUM(revenue_usd) as total FROM sales_data").collect()[0]['TOTAL']
    st.metric("Total Revenue", f"${total_revenue/1000000:.1f}M")

with col2:
    top_product = session.sql("""
        SELECT product_name FROM sales_data 
        GROUP BY product_name 
        ORDER BY SUM(revenue_usd) DESC LIMIT 1
    """).collect()[0]['PRODUCT_NAME']
    st.metric("Top Product", top_product)

with col3:
    avg_deal = session.sql("SELECT AVG(revenue_usd) as avg FROM sales_data").collect()[0]['AVG']
    st.metric("Avg Deal Size", f"${avg_deal:,.0f}")

---

##  Complete!

### What You Learned

1.  Create Semantic Models for Cortex Analyst
2.  Define business-friendly data models (YAML)
3.  Build conversational BI interfaces
4.  Enable natural language queries
5.  Execute generated SQL from questions

### Cortex Analyst Components

**Semantic Model (YAML):**
- Tables, columns, relationships
- Business-friendly names
- Measure definitions (aggregations)

**In Production:**
```sql
SELECT SNOWFLAKE.CORTEX.ANALYST.GENERATE_SQL(
    'semantic_model_name',
    'What were sales by region last month?'
)
```

Returns SQL that answers the question.

### Benefits

- **Self-service analytics**: Business users query data
- **No SQL knowledge needed**: Natural language interface
- **Consistent definitions**: Semantic models ensure accuracy
- **Faster insights**: Instant answers to ad-hoc questions

### Next Steps

- Deploy full Cortex Analyst service
- Add more complex semantic models
- Integrate with Streamlit for production chatbot
- Add authentication and access controls

### Resources

- [Cortex Analyst Docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
- [Semantic Models Guide](https://docs.snowflake.com/en/user-guide/snowflake-cortex/semantic-model-spec)

---

**Congratulations!** You've built a complete conversational BI chatbot with Cortex Analyst.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `CHATBOT_BI_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS CHATBOT_BI_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;